# Analysis of pp Events: Event Activity Observables

Firstly, we load libraries and enable multi-threading, which allows event processing across multiple CPU cores.

In [4]:
import ROOT
import numpy as np

ROOT.EnableImplicitMT() # enable multi-threading

We connect RDataFrame to the TTree in our created events.root and events_smeared.root files.

In [5]:
# we must select the data we want to work with by their seed
seed = 67 

dfTrue = ROOT.RDataFrame("events", f"data/events{seed}.root") # connect RDataFrame to the TTree in events.root
dfSmeared = ROOT.RDataFrame("events_smeared", f"data/events{seed}_smeared.root") # connect RDataFrame to the TTree in events_smeared.root

# define a data frame where true observables are replaced with the smeared ones so that we can write simple for loops
dfOnlySmeared = (
    dfSmeared.Redefine("pT", "pTSmeared")
             .Redefine("eta", "etaSmeared")
             .Redefine("phi", "phiSmeared")
)                 

# create a data frame dictionary which we can loop over
dataFrames = {
    'True': dfTrue,
    'Smeared': dfOnlySmeared
}

### Multiplicity Distribution

In [6]:
# loop over data frames
for label, df in dataFrames.items():
    dfMult = df.Define("Nch", "pT.size()") # define the multiplicity column

    histoNch =  dfMult.Histo1D((f"histoNch{label}", label + " Multiplicity;N_{ch};Normalized Events", 70, 0, 70), "Nch") # histogram of the multiplicity distribution
    histoNormalizedNch = histoNch.GetValue() # get the histogram object from the RDataFrame
    canvasMult =  ROOT.TCanvas(f"canvasMult{label}", f"{label} Multiplicity", 800, 600) # create a canvas to draw the histogram
    canvasMult.SetLogy() # set logarithmic scale for y-axis

    if histoNch.Integral() > 0: # check if the histogram has entries to avoid division by zero
        histoNormalizedNch.Scale(1.0 / histoNch.Integral()) # normalize the histogram
    histoNormalizedNch.SetLineColor(ROOT.kRed + 1) # set line color to slightly darker (+1) red
    histoNormalizedNch.SetFillColorAlpha(ROOT.kRed, 0.1) # set fill color to red with 10% transparency (alpha = 0.1)

    histoNormalizedNch.Draw("HIST") # draw the histogram
    canvasMult.SaveAs(f"img/multiplicity{label}.pdf") # save the canvas as a PDF file

Info in <TCanvas::Print>: pdf file img/multiplicityTrue.pdf has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvasMultSmeared
Info in <TCanvas::Print>: pdf file img/multiplicitySmeared.pdf has been created


### Multiplicity Response Matrix

In [7]:
dfMultiplicityRM = (
    dfSmeared.Define("NchTrue", "pT.size()") # define true multiplicity
             .Define("NchSmeared", "pTSmeared.size()") # define smeared multiplicity
)

histoNchRM = dfMultiplicityRM.Histo2D(("histoNchRM", "Multiplicity Response Matrix;N_{ch}^{MC};N_{ch}^{rec}", 70, 0, 70, 70, 0, 70), "NchTrue", "NchSmeared").GetValue() # 2D histogram with true Nch on the x-axis and smeared Nch on the y-axis

canvasNchRM = ROOT.TCanvas("canvasNchRM", "Multiplicity Response Matrix", 800, 600)
canvasNchRM.SetLogz()

histoNchRM.Draw("COLZ")
canvasNchRM.SaveAs("img/multRM.pdf")

Info in <TCanvas::Print>: pdf file img/multRM.pdf has been created


### Unweighted Spherocity $S_0^{p_\mathrm{T} = 1}$ Distribution

calculated only for $N_\mathrm{ch}(|\eta| < 1) > 10$

$$
S_0^{p_\mathrm{T} = 1} = \frac{\pi^2}{4} \min_{\hat{n}} \left( \frac{\sum_i |\hat{u}_{\mathrm{T}, i} \times \hat{n}|}{N_\mathrm{trks}} \right)^2
$$

In [8]:
# USING ONLY PYTHON AND AVOIDING C++ (sufficient for small data sets)

# define px, py using pT, phi; multiplicity as the number of pT values in each event; ux, uy using px, py, pT
df_with_px_py_Nch_ux_uy = (
    dfTrue.Define("px", "pT * cos(phi)")
          .Define("py", "pT * sin(phi)")
          .Define("Nch", "pT.size()")
          .Define("ux", "px / pT")
          .Define("uy", "py / pT")
)

# extract px, py, pT. and multiplicity from the RDataFrame into a dictionary of NumPy arrays -- this loads the data into RAM!!! BE CAREFUL WITH LARGE DATASETS --> better option is to write a function in C++ inside PyROOT as Gemini showed me, but since I'm learning, I will do it this way for now
data = df_with_px_py_Nch_ux_uy.AsNumpy(["ux", "uy", "Nch"])

# define each NumPy arrays separately
uxArr = data["ux"]
uyArr = data["uy"]
NchArr = data["Nch"]

# define the function to calculate the unweighted transverse spherocity
def calc_S0pT1_slow(uxEvent, uyEvent, NchEvent):

    # Since the minimizing axis \hat{n} is always collinear with one of the pT vectors in the event, we can calculate the sum for each pT vector and find the minimum. This is much faster than performing a numerical minimization. 
    absCrossProductMatrix = np.abs(uxEvent[:, np.newaxis]*uyEvent - uyEvent[:, np.newaxis]*uxEvent) # calculate the absolute value of the cross product of each pair of unit pT vectors utilizing [:, np.newaxis], which turns a 1D array into a column vector
    sums = np.sum(absCrossProductMatrix, axis=0) # calculate the sums for each axis (pT vector)
    minSum = np.min(sums) # find the minimum sum

    return np.pi**2 / 4 * (minSum / NchEvent)**2 # calculate and return the unweighted transverse spherocity

# prepare ROOT histogram
canvas_S0pT1_slow = ROOT.TCanvas("canvas_S0pT1_slow", "S0pT1_slow", 800, 600) # create a canvas to draw the histogram
histo_S0pT1_slow = ROOT.TH1F("histoS0pT1_slow", "Unweighted Transverse Spherocity;S_{0}^{p_{T}=1};Normalized Events", 100, 0, 1) # create a histogram for S0pT1 distribution

# event loop
for iEvent in range(len(NchArr)):
    # define observable arrays for each event
    uxEvent = uxArr[iEvent]
    uyEvent = uyArr[iEvent]
    NchEvent = NchArr[iEvent]

    if not NchEvent > 10: # select events with multiplicity > 10
        continue
    else:
        S0pT1_slow = calc_S0pT1_slow(uxEvent, uyEvent, NchEvent) # calculate the unweighted transverse spherocity for the event
        histo_S0pT1_slow.Fill(S0pT1_slow) # fill the histogram

# customize the histogram and save it
canvas_S0pT1_slow.SetLogy() # set logarithmic scale for y-axis

if histo_S0pT1_slow.Integral() > 0: # check if the histogram is not empty to avoid division by zero
    histo_S0pT1_slow.Scale(1.0 / histo_S0pT1_slow.Integral()) # normalize the histogram
histo_S0pT1_slow.SetLineColor(ROOT.kRed + 1) # set line color to slightly darker (+1) red
histo_S0pT1_slow.SetFillColorAlpha(ROOT.kRed, 0.1) # set fill color to red with 10% transparency (alpha = 0.1)

histo_S0pT1_slow.Draw("HIST") # draw the histogram
canvas_S0pT1_slow.SaveAs("img/S0pT1_slow.pdf") # save the canvas as a PDF file

Info in <TCanvas::Print>: pdf file img/S0pT1_slow.pdf has been created


In [9]:
# USING C++ IN PyROOT

# I wrote the calc_S0pT1(px, py, pT, Nch) function in C++, now we need to tell ROOT to load it and compile it. 
ROOT.gInterpreter.ProcessLine('#include "cpp/S0pT1.cpp"')

# loop over data frames
for label, df in dataFrames.items():
    dfWithPxPyNchUxUy = (
        df.Define("px", "pT * cos(phi)")
          .Define("py", "pT * sin(phi)")
          .Define("Nch", "pT.size()")
          .Define("ux", "px / pT")
          .Define("uy", "py / pT")
    )

    dfS0pT1 = (
        dfWithPxPyNchUxUy.Filter("Nch > 10", "multiplicity cut") # filter only events with multiplicity greater than 10
                         .Define("S0pT1", "calc_S0pT1(ux, uy, Nch)") # define the unweighted transverse spherocity column
    )

    histoS0pT1 = dfS0pT1.Histo1D((f"histoS0pT1{label}", label + " Unweighted Transverse Spherocity;S_{0}^{p_{T} = 1};Normalized Events", 100, 0, 1), "S0pT1") # histogram of the unweighted spherocity distribution
    histoNormalizedS0pT1 = histoS0pT1.GetValue() # get the histogram object from the RDataFrame
    canvasS0pT1 = ROOT.TCanvas(f"canvasS0pT1{label}", f"{label} S0pT1", 800, 600) # create a canvas to draw the histogram
    canvasS0pT1.SetLogy() # set logarithmic scale for y-axis

    if histoS0pT1.Integral() > 0: # check if the histogram has entries to avoid division by zero
        histoNormalizedS0pT1.Scale(1.0 / histoS0pT1.Integral()) # normalize the histogram
    histoNormalizedS0pT1.SetLineColor(ROOT.kRed + 1) # set line color to slightly darker (+1) red
    histoNormalizedS0pT1.SetFillColorAlpha(ROOT.kRed, 0.1) # set fill color to red with 10% transparency (alpha = 0.1)

    histoNormalizedS0pT1.Draw("HIST") # draw the histogram
    canvasS0pT1.SaveAs(f"img/S0pT1{label}.pdf") # save the canvas as a PDF file

Info in <TCanvas::Print>: pdf file img/S0pT1True.pdf has been created
Info in <TCanvas::Print>: pdf file img/S0pT1Smeared.pdf has been created


### Unweighted Spehrocity Response Matrix

In [10]:
ROOT.gInterpreter.ProcessLine('#include "cpp/S0pT1.cpp"') # load the calc_S0pT1 function

dfSpherocityRM = ( # ted me nenapada, jak to zefektivnit, tak to zatim napisu takto
    dfSmeared.Define("px", "pT * cos(phi)")
             .Define("py", "pT * sin(phi)")
             .Define("NchTrue", "pT.size()")
             .Define("ux", "px / pT")
             .Define("uy", "py / pT")
             .Define("pxSmeared", "pTSmeared * cos(phiSmeared)")
             .Define("pySmeared", "pTSmeared * sin(phiSmeared)")
             .Define("NchSmeared", "pTSmeared.size()")
             .Define("uxSmeared", "pxSmeared / pTSmeared")
             .Define("uySmeared", "pySmeared / pTSmeared")
             .Filter("NchTrue > 10", "true multiplicity cut")
             .Filter("NchSmeared > 10", "smeared multiplicity cut")
             .Define("S0pT1True", "calc_S0pT1(ux, uy, NchTrue)")
             .Define("S0pT1Smeared", "calc_S0pT1(uxSmeared, uySmeared, NchSmeared)")
)


histoSpherocityRM = dfSpherocityRM.Histo2D(("histoSpherocityRM", "Spherocity Response Matrix;S_{0}^{p_{T}=1, MC};S_{0}^{p_{T}=1, rec}", 100, 0, 1, 100, 0, 1), "S0pT1True", "S0pT1Smeared").GetValue() # 2D histogram with true unweighted spherocity on the x-axis and smeared unweighted spherocity on the y-axis

canvasSpherocityRM = ROOT.TCanvas("canvasSpherocityRM", "Spherocity Response Matrix", 800, 600)
canvasSpherocityRM.SetLogz()

histoSpherocityRM.Draw("COLZ")
canvasSpherocityRM.SaveAs("img/S0pT1RM.pdf")

In file included from input_line_181:1:
./cpp/S0pT1.cpp:6:8: error: redefinition of 'calc_S0pT1'
double calc_S0pT1(const ROOT::RVec<double>& uxEvent, // const is a safety lock to ensure that the function only reads data and does not change it
       ^
./cpp/S0pT1.cpp:6:8: note: previous definition is here
double calc_S0pT1(const ROOT::RVec<double>& uxEvent, // const is a safety lock to ensure that the function only reads data and does not change it
       ^
Info in <TCanvas::Print>: pdf file img/S0pT1RM.pdf has been created
